# Prometheus Integration Guide for the Iris Model

This guide maps the repository's actual Iris model API metrics to Prometheus and Grafana. The service is built and released by GitHub Actions; run it locally using the README's pull-and-run instructions. Metrics in this guide match `src/monitoring_service.py`.

## 1. Start the local monitoring stack
In PowerShell, from the repository root, set the image tag you pulled and start the services:

```powershell
$env:MODEL_IMAGE = "ghcr.io/OWNER/REPOSITORY:sha-<commit>"
docker compose up -d --wait
```

The Iris API listens on port 8000, Prometheus on 9090, and Grafana on 3000.

## 2. Send Iris prediction requests
The API expects the four Iris features in this order: `sepal_length`, `sepal_width`, `petal_length`, `petal_width`. Send requests to `POST /invocations` with the MLflow `dataframe_split` format. The README contains copy-ready PowerShell examples, including shifted values to fill the 100-row monitoring window.

## 3. Metrics exposed by the API

Open `http://localhost:8000/metrics`. The service exports:

| Metric | Type | Meaning |
|---|---|---|
| `model_requests_total{outcome="success"}` | Counter | Successful requests |
| `model_requests_total{outcome="error"}` | Counter | Failed requests |
| `model_request_duration_seconds` | Histogram | End-to-end prediction request latency |
| `model_predictions_total{prediction="0"}` | Counter | Predictions by Iris class ID |
| `model_feature_psi{feature="petal_length"}` | Gauge | Recent-window PSI for a feature vs. Iris reference |

Class IDs are scikit-learn Iris targets: 0=setosa, 1=versicolor, 2=virginica.

## 4. Useful PromQL queries

Paste these into `http://localhost:9090`:

```promql
sum(rate(model_requests_total{outcome="success"}[5m]))
```

```promql
sum(rate(model_requests_total{outcome="error"}[5m]))
/
clamp_min(sum(rate(model_requests_total[5m])), 1)
```

```promql
histogram_quantile(0.95, sum by (le) (rate(model_request_duration_seconds_bucket[5m])))
```

```promql
model_feature_psi
```

Use `model_predictions_total` to graph the count rate by predicted Iris class. PSI is only calculated after at least 100 request rows have entered the rolling window.

## 5. Grafana dashboard and alerts
Open <http://localhost:3000> and sign in with the default local credentials shown in the README. The provisioned Model Monitoring dashboard charts request rate, errors, p95 latency, and feature PSI. Prometheus loads the alert rules from `alert_rules.yml`. For production, change local credentials and tune alert thresholds against observed Iris-service traffic.

## 6. What is and is not monitored
The API provides model request outcomes, latency, prediction counts, and feature-distribution PSI. It does not collect host CPU/memory or distributed traces. Add an infrastructure exporter for host resource metrics and OpenTelemetry instrumentation plus a trace backend when those signals are required. Iris drift in the companion notebook is a controlled simulation, not real-world production data.

In [ ]:
# Quick local connectivity check (run after starting Docker Compose)
import urllib.request

for url in ("http://localhost:8000/healthz", "http://localhost:8000/metrics", "http://localhost:9090/-/ready"):
    with urllib.request.urlopen(url, timeout=5) as response:
        print(response.status, url)